# Chapter 1 — Introduction: Scientific Figures

Figures for Chapter 1 illustrating the historical evolution of equations of
state, the limitations of classical cubic EoS for associating fluids, and
the scope of systems where CPA provides significant improvements.

**Figures:**
1. Timeline of equation-of-state development
2. Water density: cubic EoS failure vs CPA
3. Industries and systems requiring association models

In [1]:
import importlib, subprocess, sys
try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False); ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
except Exception:
    try: import neqsim
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
print(f"NeqSim mode: {NEQSIM_MODE}")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim


NeqSim mode: pip


In [2]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = ["#2171b5", "#e6550d", "#31a354", "#756bb1", "#e7298a", "#66a61e"]
BLUE, ORANGE, GREEN, PURPLE, PINK, LIME = PALETTE
plt.rcParams.update({
    "font.family": "serif", "font.size": 9, "axes.labelsize": 10,
    "axes.titlesize": 10, "legend.fontsize": 8, "xtick.labelsize": 8,
    "ytick.labelsize": 8, "xtick.direction": "in", "ytick.direction": "in",
    "axes.linewidth": 0.6, "lines.linewidth": 1.2, "grid.linewidth": 0.3,
    "grid.alpha": 0.4, "savefig.dpi": 300, "figure.dpi": 150,
})
FIGURES_DIR = Path("../figures"); FIGURES_DIR.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")

## Figure 1.1 — Timeline of Equation-of-State Development

In [3]:
events = [
    (1873, "van der Waals", "First cubic EoS"),
    (1949, "Redlich\u2013Kwong", "Improved temperature dep."),
    (1972, "Soave (SRK)", "Acentric factor"),
    (1976, "Peng\u2013Robinson", "Better liquid density"),
    (1984, "Wertheim TPT1", "Association theory"),
    (1988, "Chapman (SAFT)", "SAFT EoS family"),
    (1996, "Kontogeorgis (CPA)", "Cubic + Association"),
    (2001, "Gross & Sadowski", "PC-SAFT"),
    (2006, "Folas et al.", "s-CPA (simplified)"),
]

fig, ax = plt.subplots(figsize=(7.0, 2.5))
years = [e[0] for e in events]
colors = [BLUE]*4 + [ORANGE]*2 + [GREEN]*3

ax.scatter(years, [0]*len(years), c=colors, s=50, zorder=5, edgecolors="white", linewidths=0.5)
ax.plot([min(years)-5, max(years)+5], [0, 0], 'k-', linewidth=0.8, zorder=1)

for i, (yr, name, note) in enumerate(events):
    side = 1 if i % 2 == 0 else -1
    ax.annotate(f"{yr}: {name}", xy=(yr, 0),
                xytext=(yr, side * 0.35),
                fontsize=6.5, ha="center", va="center",
                arrowprops=dict(arrowstyle="-", color="grey", lw=0.4),
                bbox=dict(boxstyle="round,pad=0.15", facecolor="#f0f0f0", edgecolor="grey", linewidth=0.4))

ax.set_xlim(1865, 2015)
ax.set_ylim(-0.7, 0.7)
ax.set_xlabel("Year")
ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_title("Evolution of Equations of State")

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor=BLUE, label="Classical cubic"),
    Patch(facecolor=ORANGE, label="Association theory"),
    Patch(facecolor=GREEN, label="CPA family"),
], loc="upper left", fontsize=6, frameon=True)

fig.tight_layout()
save(fig, "fig_ch01_01_eos_timeline.png")

Saved: fig_ch01_01_eos_timeline.png


## Figure 1.2 — Why Cubic EoS Fail for Water

NeqSim comparison: SRK vs CPA for liquid water density.

In [4]:
if NEQSIM_MODE == "pip":
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    SystemPrEos = jneqsim.thermo.system.SystemPrEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
else:
    from neqsim import jneqsim
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    SystemPrEos = jneqsim.thermo.system.SystemPrEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

temps_C = np.arange(10, 350, 15)
models = {
    "SRK": (SystemSrkEos, "classic", ORANGE, "--"),
    "PR": (SystemPrEos, "classic", PURPLE, "-."),
    "CPA": (SystemSrkCPAstatoil, 10, BLUE, "-"),
}
results = {name: {"T": [], "rho": []} for name in models}

for T_C in temps_C:
    for name, (cls, mr, col, ls) in models.items():
        try:
            f = cls(273.15 + float(T_C), 1.0)
            f.addComponent("water", 1.0)
            if isinstance(mr, int): f.setMixingRule(mr)
            else: f.setMixingRule(mr)
            ops = ThermodynamicOperations(f)
            ops.bubblePointPressureFlash(False)
            f.initProperties()
            phase_name = "aqueous" if name == "CPA" else "oil"
            rho = float(f.getPhase(phase_name).getDensity("kg/m3"))
            results[name]["T"].append(float(T_C))
            results[name]["rho"].append(rho)
        except Exception:
            pass

nist_T = [25, 50, 100, 150, 200, 250, 300, 350]
nist_rho = [997.0, 988.0, 958.4, 917.0, 864.7, 799.2, 712.4, 574.7]

fig, ax = plt.subplots(figsize=(3.5, 2.8))
for name, (cls, mr, col, ls) in models.items():
    ax.plot(results[name]["T"], results[name]["rho"], color=col, linestyle=ls,
            linewidth=1.5, label=name)
ax.scatter(nist_T, nist_rho, color="black", s=25, zorder=5, label="NIST",
           edgecolors="black", linewidths=0.5)
ax.set_xlabel("Temperature (\u00b0C)")
ax.set_ylabel("Liquid density (kg/m\u00b3)")
ax.set_title("Water: CPA vs classical cubic")
ax.legend(frameon=True); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch01_02_water_density_comparison.png")

Saved: fig_ch01_02_water_density_comparison.png


## Figure 1.3 — Industrial Systems Requiring Association Models

Schematic showing the breadth of systems where CPA significantly
outperforms classical cubic EoS.

In [5]:
categories = [
    ("Gas Processing", ["TEG dehydration", "MEG injection", "Hydrate prediction"]),
    ("CO\u2082 Capture", ["CO\u2082\u2013water VLE", "Amine absorption", "CCS transport"]),
    ("Petroleum", ["Water\u2013oil\u2013gas", "Produced water", "Asphaltenes"]),
    ("Chemical", ["Alcohol mixtures", "Acid systems", "Polymer\u2013solvent"]),
]

fig, ax = plt.subplots(figsize=(7.0, 3.0))
colors = [BLUE, ORANGE, GREEN, PURPLE]

y_pos = 0
for i, (cat, items) in enumerate(categories):
    ax.barh(y_pos, 1.0, height=0.6, color=colors[i], alpha=0.8)
    ax.text(-0.05, y_pos, cat, ha="right", va="center", fontsize=9, fontweight="bold")
    for j, item in enumerate(items):
        ax.text(0.1 + j * 0.33, y_pos, item, ha="left", va="center", fontsize=7, color="white")
    y_pos -= 1

ax.set_xlim(-0.5, 1.1)
ax.set_ylim(y_pos + 0.3, 0.6)
ax.axis("off")
ax.set_title("Industrial Systems Requiring Association Models", fontsize=10, fontweight="bold")
fig.tight_layout()
save(fig, "fig_ch01_03_industrial_systems.png")

Saved: fig_ch01_03_industrial_systems.png
